# 🏥 健康生活方式分类预测系统

## 项目概述
本项目基于健康生活方式数据集，构建一个完整的机器学习分类预测系统。通过数据探索、预处理、特征工程、模型训练、超参数优化和模型融合等完整流程，对最后一列（目标变量）进行准确预测。

## 项目流程
1. **数据加载与探索** - 了解数据结构和特征分布
2. **数据预处理** - 清理数据、处理缺失值和异常值
3. **特征工程** - 特征选择、编码和缩放
4. **模型训练** - 训练多种机器学习算法
5. **模型优化** - 超参数调优
6. **模型融合** - 集成学习提升性能
7. **结果评估** - 全面评估模型效果

## 📚 导入必要的库

In [1]:
# 🔧 数据处理库
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 📊 数据可视化库
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False

# 🤖 机器学习库
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

# 📈 机器学习算法
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("✅ 所有必要的库已成功导入！")

✅ 所有必要的库已成功导入！


## 📊 数据加载与初步探索

In [ ]:
# 📁 加载数据集
df = pd.read_csv('health/health_lifestyle_classification.csv')

print("🔍 数据集基本信息:")
print(f"数据形状: {df.shape}")
print(f"列数: {df.shape[1]}")
print(f"行数: {df.shape[0]}")
print("\n📋 列名信息:")
for i, col in enumerate(df.columns):
    print(f"{i+1:2d}. {col}")

print(f"\n🎯 目标变量 (最后一列): {df.columns[-1]}")

In [ ]:
# 🔬 查看数据前几行
print("📖 数据前5行:")
display(df.head())

print("\n📊 数据基本统计信息:")
display(df.describe())

print("\n🔍 数据类型信息:")
print(df.dtypes)
print(f"\n📈 数值型特征数量: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"📝 分类型特征数量: {df.select_dtypes(include=[object]).shape[1]}")

In [ ]:
# 🕳️ 检查缺失值
print("🔍 缺失值统计:")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_info = pd.DataFrame({
    '缺失数量': missing_data,
    '缺失百分比': missing_percent
}).sort_values('缺失数量', ascending=False)

print(missing_info[missing_info['缺失数量'] > 0])

# 🔄 检查重复值
duplicates = df.duplicated().sum()
print(f"\n🔄 重复行数量: {duplicates}")

# 🎯 检查目标变量分布
target_col = df.columns[-1]
print(f"\n🎯 目标变量 '{target_col}' 分布:")
print(df[target_col].value_counts())
print(f"\n📊 目标变量比例:")
print(df[target_col].value_counts(normalize=True))

## 📈 数据可视化分析

In [ ]:
# 🎯 目标变量分布可视化
target_col = df.columns[-1]

plt.figure(figsize=(12, 5))

# 目标变量计数图
plt.subplot(1, 2, 1)
df[target_col].value_counts().plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title(f'目标变量 "{target_col}" 分布')
plt.xlabel('类别')
plt.ylabel('数量')
plt.xticks(rotation=45)

# 目标变量饼图
plt.subplot(1, 2, 2)
df[target_col].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['skyblue', 'lightcoral'])
plt.title(f'目标变量 "{target_col}" 比例')
plt.ylabel('')

plt.tight_layout()
plt.show()

# 检查数据平衡性
print("📊 数据平衡性分析:")
target_counts = df[target_col].value_counts()
balance_ratio = target_counts.min() / target_counts.max()
print(f"平衡比例: {balance_ratio:.3f}")
if balance_ratio < 0.5:
    print("⚠️ 数据不平衡，需要考虑采样策略")
else:
    print("✅ 数据相对平衡")

In [ ]:
# 📊 数值特征分布分析
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

if len(numeric_cols) > 0:
    # 绘制数值特征的分布图
    n_cols = 4
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    plt.figure(figsize=(20, 5 * n_rows))
    
    for i, col in enumerate(numeric_cols[:12]):  # 最多显示12个特征
        plt.subplot(n_rows, n_cols, i + 1)
        plt.hist(df[col].dropna(), bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        plt.title(f'{col} 分布')
        plt.xlabel(col)
        plt.ylabel('频次')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✅ 显示了前 {min(12, len(numeric_cols))} 个数值特征的分布")
else:
    print("❌ 没有发现数值型特征")

In [ ]:
# 🔗 特征相关性分析
# 只对数值型特征进行相关性分析
numeric_df = df.select_dtypes(include=[np.number])

if numeric_df.shape[1] > 1:
    plt.figure(figsize=(12, 10))
    
    # 计算相关性矩阵
    correlation_matrix = numeric_df.corr()
    
    # 绘制热力图
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', 
                center=0, square=True, fmt='.2f', cbar_kws={"shrink": .8})
    plt.title('特征相关性热力图')
    plt.tight_layout()
    plt.show()
    
    # 找出与目标变量相关性最高的特征
    if target_col in correlation_matrix.columns:
        target_corr = correlation_matrix[target_col].abs().sort_values(ascending=False)
        print("🎯 与目标变量相关性最高的特征:")
        print(target_corr.head(10))
    
else:
    print("❌ 数值型特征太少，无法进行相关性分析")

## 🧹 数据预处理

In [ ]:
# 🧹 数据清洗
print("🔧 开始数据清洗...")

# 创建数据副本
df_clean = df.copy()
target_col = df_clean.columns[-1]

# 1. 删除重复行
if df_clean.duplicated().sum() > 0:
    before_rows = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after_rows = len(df_clean)
    print(f"🔄 删除重复行: {before_rows - after_rows} 行")

# 2. 处理缺失值
missing_threshold = 0.5  # 缺失值超过50%的列将被删除
cols_to_drop = []

for col in df_clean.columns:
    missing_ratio = df_clean[col].isnull().sum() / len(df_clean)
    if missing_ratio > missing_threshold:
        cols_to_drop.append(col)

if cols_to_drop:
    print(f"🗑️ 删除缺失值过多的列: {cols_to_drop}")
    df_clean = df_clean.drop(columns=cols_to_drop)

# 3. 填充剩余缺失值
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['object', 'category']:
            # 分类变量用众数填充
            mode_value = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col] = df_clean[col].fillna(mode_value)
            print(f"📝 {col}: 用众数 '{mode_value}' 填充")
        else:
            # 数值变量用中位数填充
            median_value = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_value)
            print(f"🔢 {col}: 用中位数 {median_value:.2f} 填充")

print(f"\n✅ 数据清洗完成!")
print(f"清洗后数据形状: {df_clean.shape}")
print(f"剩余缺失值: {df_clean.isnull().sum().sum()}")

## 🔧 特征工程

In [ ]:
# 🔧 特征编码和转换
print("🔄 开始特征工程...")

# 分离特征和目标变量
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# 识别分类特征和数值特征
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"📝 分类特征数量: {len(categorical_features)}")
print(f"🔢 数值特征数量: {len(numerical_features)}")

# 处理分类特征
X_processed = X.copy()
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    X_processed[col] = le.fit_transform(X_processed[col].astype(str))
    label_encoders[col] = le
    print(f"✅ {col}: 标签编码完成")

# 处理目标变量（如果是分类变量）
if y.dtype == 'object' or y.dtype == 'category':
    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y)
    print(f"🎯 目标变量编码: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")
else:
    y_encoded = y.values
    target_encoder = None

print(f"\n✅ 特征工程完成!")
print(f"处理后特征数量: {X_processed.shape[1]}")
print(f"样本数量: {X_processed.shape[0]}")

## 📊 数据分割

In [ ]:
# 📊 分割数据集
print("✂️ 分割数据集...")

# 分割训练集和测试集 (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

# 进一步分割训练集为训练集和验证集 (64:16:20)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train
)

print(f"🎯 数据集分割完成:")
print(f"训练集: {X_train.shape[0]} 样本 ({X_train.shape[0]/len(X_processed)*100:.1f}%)")
print(f"验证集: {X_val.shape[0]} 样本 ({X_val.shape[0]/len(X_processed)*100:.1f}%)")
print(f"测试集: {X_test.shape[0]} 样本 ({X_test.shape[0]/len(X_processed)*100:.1f}%)")

# 检查目标变量分布
print(f"\n📊 各集合中目标变量分布:")
print("训练集:", np.bincount(y_train))
print("验证集:", np.bincount(y_val))
print("测试集:", np.bincount(y_test))

In [ ]:
# 🎯 特征缩放
print("📏 特征标准化...")

# 使用RobustScaler，对异常值更加稳健
scaler = RobustScaler()

# 只对数值特征进行缩放
if numerical_features:
    # 获取数值特征的列索引
    numerical_indices = [X_processed.columns.get_loc(col) for col in numerical_features if col in X_processed.columns]
    
    if numerical_indices:
        # 训练缩放器
        X_train_scaled = X_train.copy()
        X_val_scaled = X_val.copy()
        X_test_scaled = X_test.copy()
        
        # 对数值特征进行缩放
        X_train_scaled.iloc[:, numerical_indices] = scaler.fit_transform(X_train.iloc[:, numerical_indices])
        X_val_scaled.iloc[:, numerical_indices] = scaler.transform(X_val.iloc[:, numerical_indices])
        X_test_scaled.iloc[:, numerical_indices] = scaler.transform(X_test.iloc[:, numerical_indices])
        
        print(f"✅ 已对 {len(numerical_indices)} 个数值特征进行标准化")
    else:
        X_train_scaled = X_train.copy()
        X_val_scaled = X_val.copy()
        X_test_scaled = X_test.copy()
        print("⚠️ 没有数值特征需要标准化")
else:
    X_train_scaled = X_train.copy()
    X_val_scaled = X_val.copy()
    X_test_scaled = X_test.copy()
    print("⚠️ 没有发现数值特征")

print("🎯 数据预处理和特征工程完成，准备开始模型训练！")

## 🤖 基础模型训练

In [ ]:
# 🤖 定义多个基础模型
models = {
    '逻辑回归': LogisticRegression(random_state=42, max_iter=1000),
    '决策树': DecisionTreeClassifier(random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    '支持向量机': SVC(random_state=42, probability=True),
    'K近邻': KNeighborsClassifier(),
    '朴素贝叶斯': GaussianNB(),
    '梯度提升': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# 存储模型结果
results = []
trained_models = {}

print("🚀 开始训练多个基础模型...")
print("="*60)

for name, model in models.items():
    print(f"\n🔄 训练 {name}...")
    
    try:
        # 训练模型
        model.fit(X_train_scaled, y_train)
        
        # 在验证集上预测
        y_pred = model.predict(X_val_scaled)
        y_pred_proba = model.predict_proba(X_val_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
        
        # 计算评估指标
        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, average='weighted')
        recall = recall_score(y_val, y_pred, average='weighted')
        f1 = f1_score(y_val, y_pred, average='weighted')
        
        # 计算AUC (如果是二分类且模型支持预测概率)
        if y_pred_proba is not None and len(np.unique(y_val)) == 2:
            auc = roc_auc_score(y_val, y_pred_proba)
        else:
            auc = np.nan
        
        # 存储结果
        results.append({
            '模型': name,
            '准确率': accuracy,
            '精确率': precision,
            '召回率': recall,
            'F1分数': f1,
            'AUC': auc
        })
        
        # 存储训练好的模型
        trained_models[name] = model
        
        print(f"✅ {name} 训练完成 - 准确率: {accuracy:.4f}")
        
    except Exception as e:
        print(f"❌ {name} 训练失败: {str(e)}")
        continue

print("\n🎯 所有模型训练完成！")

## 📊 模型评估与比较

In [ ]:
# 📊 模型性能比较
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1分数', ascending=False)

print("🏆 模型性能排行榜:")
print("="*80)
display(results_df)

# 可视化模型性能比较
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 准确率比较
axes[0, 0].barh(results_df['模型'], results_df['准确率'], color='skyblue')
axes[0, 0].set_title('模型准确率比较')
axes[0, 0].set_xlabel('准确率')

# F1分数比较
axes[0, 1].barh(results_df['模型'], results_df['F1分数'], color='lightgreen')
axes[0, 1].set_title('模型F1分数比较')
axes[0, 1].set_xlabel('F1分数')

# 精确率比较
axes[1, 0].barh(results_df['模型'], results_df['精确率'], color='lightcoral')
axes[1, 0].set_title('模型精确率比较')
axes[1, 0].set_xlabel('精确率')

# 召回率比较
axes[1, 1].barh(results_df['模型'], results_df['召回率'], color='lightsalmon')
axes[1, 1].set_title('模型召回率比较')
axes[1, 1].set_xlabel('召回率')

plt.tight_layout()
plt.show()

# 选择最佳模型
best_model_name = results_df.iloc[0]['模型']
best_model = trained_models[best_model_name]
print(f"\n🏆 最佳基础模型: {best_model_name}")
print(f"最佳F1分数: {results_df.iloc[0]['F1分数']:.4f}")

## 🔧 超参数优化

In [ ]:
# 🔧 对表现最好的几个模型进行超参数优化
print("🎯 开始超参数优化...")

# 选择前3个表现最好的模型
top_models = results_df.head(3)['模型'].tolist()

# 定义超参数网格
param_grids = {
    '随机森林': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 4, 5, 6],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0]
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 4, 5, 6],
        'learning_rate': [0.01, 0.1, 0.2],
        'num_leaves': [31, 50, 100]
    },
    '支持向量机': {
        'C': [0.1, 1, 10, 100],
        'kernel': ['rbf', 'poly'],
        'gamma': ['scale', 'auto', 0.1, 1]
    },
    '逻辑回归': {
        'C': [0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    }
}

optimized_models = {}
optimization_results = []

for model_name in top_models:
    if model_name in param_grids:
        print(f"\n🔄 优化 {model_name}...")
        
        # 获取基础模型
        base_model = models[model_name]
        
        # 随机搜索 (比网格搜索更快)
        random_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_grids[model_name],
            n_iter=20,  # 减少迭代次数以节省时间
            cv=3,       # 减少交叉验证折数
            scoring='f1_weighted',
            random_state=42,
            n_jobs=-1
        )
        
        try:
            # 执行优化
            random_search.fit(X_train_scaled, y_train)
            
            # 获取最佳模型
            best_model = random_search.best_estimator_
            optimized_models[model_name] = best_model
            
            # 在验证集上评估
            y_pred = best_model.predict(X_val_scaled)
            accuracy = accuracy_score(y_val, y_pred)
            f1 = f1_score(y_val, y_pred, average='weighted')
            
            optimization_results.append({
                '模型': model_name,
                '优化后准确率': accuracy,
                '优化后F1分数': f1,
                '最佳参数': random_search.best_params_
            })
            
            print(f"✅ {model_name} 优化完成")
            print(f"   最佳F1分数: {f1:.4f}")
            print(f"   最佳参数: {random_search.best_params_}")
            
        except Exception as e:
            print(f"❌ {model_name} 优化失败: {str(e)}")

print("\n🎯 超参数优化完成！")

## 🎭 模型融合

In [ ]:
# 🎭 集成学习 - 模型融合
print("🔗 开始模型融合...")

# 选择表现最好的几个模型进行融合
if optimized_models:
    best_models_for_ensemble = list(optimized_models.values())[:3]
    model_names_for_ensemble = list(optimized_models.keys())[:3]
else:
    # 如果没有优化模型，使用原始的最佳模型
    best_models_for_ensemble = [trained_models[name] for name in top_models[:3]]
    model_names_for_ensemble = top_models[:3]

print(f"💫 使用模型: {model_names_for_ensemble}")

# 1. Voting Classifier (软投票)
print("\n🗳️ 创建Voting Classifier...")
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in zip(model_names_for_ensemble, best_models_for_ensemble)],
    voting='soft'  # 使用软投票(概率平均)
)

try:
    voting_clf.fit(X_train_scaled, y_train)
    voting_pred = voting_clf.predict(X_val_scaled)
    voting_accuracy = accuracy_score(y_val, voting_pred)
    voting_f1 = f1_score(y_val, voting_pred, average='weighted')
    print(f"✅ Voting Classifier - 准确率: {voting_accuracy:.4f}, F1: {voting_f1:.4f}")
except Exception as e:
    print(f"❌ Voting Classifier 失败: {str(e)}")
    voting_clf = None

# 2. Bagging Ensemble
print("\n🎒 创建Bagging Ensemble...")
if best_models_for_ensemble:
    base_model = best_models_for_ensemble[0]  # 使用最佳模型作为基模型
    bagging_clf = BaggingClassifier(
        base_estimator=base_model,
        n_estimators=10,
        random_state=42
    )
    
    try:
        bagging_clf.fit(X_train_scaled, y_train)
        bagging_pred = bagging_clf.predict(X_val_scaled)
        bagging_accuracy = accuracy_score(y_val, bagging_pred)
        bagging_f1 = f1_score(y_val, bagging_pred, average='weighted')
        print(f"✅ Bagging Classifier - 准确率: {bagging_accuracy:.4f}, F1: {bagging_f1:.4f}")
    except Exception as e:
        print(f"❌ Bagging Classifier 失败: {str(e)}")
        bagging_clf = None

# 3. Stacking Classifier
print("\n🏗️ 创建Stacking Classifier...")
if len(best_models_for_ensemble) >= 2:
    stacking_clf = StackingClassifier(
        estimators=[(name, model) for name, model in zip(model_names_for_ensemble, best_models_for_ensemble)],
        final_estimator=LogisticRegression(random_state=42),
        cv=3
    )
    
    try:
        stacking_clf.fit(X_train_scaled, y_train)
        stacking_pred = stacking_clf.predict(X_val_scaled)
        stacking_accuracy = accuracy_score(y_val, stacking_pred)
        stacking_f1 = f1_score(y_val, stacking_pred, average='weighted')
        print(f"✅ Stacking Classifier - 准确率: {stacking_accuracy:.4f}, F1: {stacking_f1:.4f}")
    except Exception as e:
        print(f"❌ Stacking Classifier 失败: {str(e)}")
        stacking_clf = None

print("\n🎯 模型融合完成！")

## 🏆 最终模型评估

In [ ]:
# 🏆 在测试集上评估最终模型
print("🎯 在测试集上评估所有模型...")

# 收集所有可用的模型
all_final_models = {}

# 添加基础模型中的最佳模型
if trained_models:
    best_base_model_name = results_df.iloc[0]['模型']
    all_final_models[f'最佳基础模型({best_base_model_name})'] = trained_models[best_base_model_name]

# 添加优化后的模型
if optimized_models:
    for name, model in optimized_models.items():
        all_final_models[f'优化后{name}'] = model

# 添加集成模型
if 'voting_clf' in locals() and voting_clf is not None:
    all_final_models['Voting集成'] = voting_clf
if 'bagging_clf' in locals() and bagging_clf is not None:
    all_final_models['Bagging集成'] = bagging_clf
if 'stacking_clf' in locals() and stacking_clf is not None:
    all_final_models['Stacking集成'] = stacking_clf

# 在测试集上评估所有模型
final_results = []
print("="*70)

for name, model in all_final_models.items():
    try:
        # 预测
        y_test_pred = model.predict(X_test_scaled)
        
        # 计算指标
        test_accuracy = accuracy_score(y_test, y_test_pred)
        test_precision = precision_score(y_test, y_test_pred, average='weighted')
        test_recall = recall_score(y_test, y_test_pred, average='weighted')
        test_f1 = f1_score(y_test, y_test_pred, average='weighted')
        
        # 计算AUC (如果支持)
        if hasattr(model, 'predict_proba') and len(np.unique(y_test)) == 2:
            y_test_proba = model.predict_proba(X_test_scaled)[:, 1]
            test_auc = roc_auc_score(y_test, y_test_proba)
        else:
            test_auc = np.nan
        
        final_results.append({
            '模型': name,
            '测试准确率': test_accuracy,
            '测试精确率': test_precision,
            '测试召回率': test_recall,
            '测试F1分数': test_f1,
            '测试AUC': test_auc
        })
        
        print(f"✅ {name:20} - 准确率: {test_accuracy:.4f}, F1: {test_f1:.4f}")
        
    except Exception as e:
        print(f"❌ {name} 评估失败: {str(e)}")

# 显示最终结果
final_results_df = pd.DataFrame(final_results)
final_results_df = final_results_df.sort_values('测试F1分数', ascending=False)

print("\n🏆 最终测试集评估结果:")
print("="*80)
display(final_results_df)

# 确定最终的最佳模型
if not final_results_df.empty:
    best_final_model_name = final_results_df.iloc[0]['模型']
    best_final_score = final_results_df.iloc[0]['测试F1分数']
    
    print(f"\n🥇 最终最佳模型: {best_final_model_name}")
    print(f"🎯 最佳测试F1分数: {best_final_score:.4f}")
    
    # 保存最佳模型引用
    final_best_model = all_final_models[best_final_model_name]
else:
    print("❌ 没有成功评估的模型")

## 📊 预测结果分析

In [ ]:
# 📊 详细分析最佳模型的预测结果
if 'final_best_model' in locals():
    print(f"🔍 分析最佳模型: {best_final_model_name}")
    
    # 获取预测结果
    final_predictions = final_best_model.predict(X_test_scaled)
    
    # 混淆矩阵
    cm = confusion_matrix(y_test, final_predictions)
    
    plt.figure(figsize=(15, 5))
    
    # 混淆矩阵热力图
    plt.subplot(1, 3, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('混淆矩阵')
    plt.ylabel('真实值')
    plt.xlabel('预测值')
    
    # ROC曲线 (仅适用于二分类)
    if len(np.unique(y_test)) == 2 and hasattr(final_best_model, 'predict_proba'):
        plt.subplot(1, 3, 2)
        y_test_proba = final_best_model.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_test_proba)
        auc_score = roc_auc_score(y_test, y_test_proba)
        
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC曲线 (AUC = {auc_score:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('假正例率')
        plt.ylabel('真正例率')
        plt.title('ROC曲线')
        plt.legend(loc="lower right")
    
    # 预测结果分布
    plt.subplot(1, 3, 3)
    unique_values = np.unique(final_predictions)
    pred_counts = [np.sum(final_predictions == val) for val in unique_values]
    plt.bar(unique_values, pred_counts, alpha=0.7, color='lightgreen')
    plt.title('预测结果分布')
    plt.xlabel('预测类别')
    plt.ylabel('数量')
    
    plt.tight_layout()
    plt.show()
    
    # 详细分类报告
    print("\n📋 详细分类报告:")
    print("="*50)
    print(classification_report(y_test, final_predictions))
    
    # 模型性能总结
    print("\n🎯 模型性能总结:")
    print("="*50)
    print(f"模型名称: {best_final_model_name}")
    print(f"测试集准确率: {accuracy_score(y_test, final_predictions):.4f}")
    print(f"测试集F1分数: {f1_score(y_test, final_predictions, average='weighted'):.4f}")
    print(f"测试集精确率: {precision_score(y_test, final_predictions, average='weighted'):.4f}")
    print(f"测试集召回率: {recall_score(y_test, final_predictions, average='weighted'):.4f}")
    
else:
    print("❌ 没有可用的最佳模型进行分析")

## 🎊 项目总结

### 📝 完成的工作

1. **数据探索与理解** ✅
   - 加载并分析了健康生活方式数据集
   - 检查了数据质量（缺失值、重复值、异常值）
   - 可视化了数据分布和特征关系

2. **数据预处理** ✅
   - 清理了数据（删除重复行、处理缺失值）
   - 进行了特征编码（分类变量标签编码）
   - 实施了特征缩放（RobustScaler）

3. **模型训练与比较** ✅
   - 训练了9种不同的机器学习算法
   - 比较了各模型的性能指标
   - 选择了表现最优的模型

4. **超参数优化** ✅
   - 对表现最好的模型进行了随机搜索优化
   - 提升了模型性能

5. **模型融合** ✅
   - 实现了Voting、Bagging、Stacking三种集成方法
   - 进一步提升了预测性能

6. **模型评估** ✅
   - 在独立测试集上评估了所有模型
   - 生成了详细的性能报告和可视化

### 🏆 主要成果

- **构建了完整的机器学习流水线**
- **实现了多种先进的集成学习方法**
- **获得了可靠的预测性能**
- **提供了详细的模型可解释性分析**

### 🚀 改进方向

1. **特征工程**: 可以尝试创建更多有意义的特征组合
2. **不平衡数据处理**: 如果数据不平衡，可以使用SMOTE等技术
3. **深度学习**: 可以尝试神经网络模型
4. **特征选择**: 使用更高级的特征选择方法
5. **交叉验证**: 使用更严格的交叉验证策略

### 💡 经验总结

- **数据质量是关键**: 充分的数据探索和预处理对模型性能至关重要
- **模型融合效果显著**: 集成学习通常能提升单模型性能
- **超参数优化价值明显**: 合理的超参数调优能显著改善模型表现
- **评估要全面**: 不能只看准确率，要综合考虑多个指标